# Phase 1: Exploratory Data Analysis (EDA)

**Objective**: Familiarize with the CWRU Bearing Dataset, assess data quality, and visualize raw vibration signals.

### Key Activities:
1. **Raw Signal Visualization**: Time-domain plots of Drive End (DE) and Fan End (FE) data.
2. **Distribution Analysis**: Density plots and Boxplots for amplitude assessment.
3. **Outlier Detection**: Identifying signal anomalies and noise.
4. **Bearing Specs Integration**: Calculating characteristic frequencies based on geometry.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import loadmat

# Setup aesthetics
plt.style.use('ggplot')
sns.set_palette("viridis")

RAW_DIR = "../data/raw"
SPECS_PATH = "../data/external/bearing_specs.csv"

print("EDA Environment Ready.")

## 1. Load Bearing Specifications
Understanding the physical geometry of the bearings is crucial for calculating fault frequencies later.

In [ ]:
specs_df = pd.read_csv(SPECS_PATH)
display(specs_df)

def get_fault_frequencies(rpm, bearing_name='SKF_6205'):
    row = specs_df[specs_df['bearing_name'] == bearing_name].iloc[0]
    bd = row['ball_diameter']
    pd_val = row['pitch_diameter']
    n = row['n_balls']
    phi = 0 # Contact angle assume 0
    
    fr = rpm / 60
    bpfo = (n * fr / 2) * (1 - (bd/pd_val) * np.cos(phi))
    bpfi = (n * fr / 2) * (1 + (bd/pd_val) * np.cos(phi))
    bsf = (pd_val * fr / (2*bd)) * (1 - (bd/pd_val)**2 * np.cos(phi)**2)
    
    return {'BPFO': bpfo, 'BPFI': bpfi, 'BSF': bsf}

print("Sample Fault Frequencies at 1797 RPM (DE):", get_fault_frequencies(1797))

## 2. Visualize Raw Vibration Signals
We compare Normal state vs Inner Race fault.

In [ ]:
def load_cwru_mat(path):
    data = loadmat(path)
    # Handle varying keys
    for key in data.keys():
        if 'DE_time' in key: de = data[key].flatten()
        if 'FE_time' in key: fe = data[key].flatten()
    return de[:10000], fe[:10000] # Load first 10k points

normal_file = os.path.join(RAW_DIR, 'normal/97.mat')
fault_file = os.path.join(RAW_DIR, '12k_drive_end/105.mat') # 0.007 IR

de_norm, fe_norm = load_cwru_mat(normal_file)
de_fault, fe_fault = load_cwru_mat(fault_file)

fig, ax = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
ax[0].plot(de_norm, label='Normal DE', alpha=0.8, color='green')
ax[0].set_title('Normal Signal (Drive End)')
ax[0].legend()

ax[1].plot(de_fault, label='Inner Race Fault DE (0.007)', alpha=0.8, color='red')
ax[1].set_title('Faulty Signal (Drive End)')
ax[1].legend()

plt.xlabel('Sample Index')
plt.tight_layout()
plt.show()

## 3. Data Quality & Distribution
Check for outliers using Z-score and Boxplots.

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=[de_norm, de_fault], palette=['green', 'red'])
plt.xticks([0, 1], ['Normal', 'Fault (IR 0.007)'])
plt.title('Vibration Amplitude Distribution (Outlier Comparison)')
plt.ylabel('Acceleration (g)')
plt.show()

z_scores = (de_fault - np.mean(de_fault)) / np.std(de_fault)
outliers_count = np.sum(np.abs(z_scores) > 3)
print(f"Number of statistical outliers (Z > 3) in Fault signal: {outliers_count} ({outliers_count/len(de_fault):.2%})")

## 4. Signal Histograms
Observe the skewness and kurtosis visually.

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(de_norm, color='green', label='Normal', kde=True, stat="density", alpha=0.3)
sns.histplot(de_fault, color='red', label='Fault', kde=True, stat="density", alpha=0.3)
plt.title('Signal Histograms - PDF Comparison')
plt.legend()
plt.show()